# 🚀 POLYDIM V5 - EL FIN DEL GUSANO 2D
Este cuaderno arregla el leak de causalidad detectado en V4. 
Implementa operaciones **100% en Tensores Complejos** desde el Fourier Embedding hasta la salida.
El Laplaciano Magnético ahora usa una matriz de Adyacencia **Triangular Inferior** para forzar causalidad estricta.

In [ ]:
# ============================================================
# POLYDIM V5 - ENTRENAMIENTO TINYSHAKESPEARE (COMPLEX/CAUSAL FIX)
# ============================================================
# Fixes V5: 
# - La proyección a .real destruía la fase causal magnética en V4.
# - Todo el manifold latente ahora opera en C (Números Complejos).
# - Componentes actualizados: ComplexLinear, ComplexLayerNorm, ComplexGELU.
# - El Laplaciano preserva la asimetría temporal (GPT-like, no BERT-like).
# - Isometría FFT preserva el dominio complejo de principio a fin.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import requests
import time
import json

torch.manual_seed(1337)
np.random.seed(1337)

# -----------------------------------------------------------
# 0. DATASET
# -----------------------------------------------------------
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"[DATA] Longitud: {len(text)}, Vocab: {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# -----------------------------------------------------------
# 1. CORE COMPLEJO (V5)
# -----------------------------------------------------------

class ComplexLinear(nn.Module):
    """Proyección lineal en el dominio complejo C."""
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.fc_r = nn.Linear(in_features, out_features, bias=bias)
        self.fc_i = nn.Linear(in_features, out_features, bias=bias)
        
    def forward(self, x):
        real = self.fc_r(x.real) - self.fc_i(x.imag)
        imag = self.fc_r(x.imag) + self.fc_i(x.real)
        return torch.complex(real, imag)

class ComplexLayerNorm(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.ln_r = nn.LayerNorm(D)
        self.ln_i = nn.LayerNorm(D)
        
    def forward(self, x):
        return torch.complex(self.ln_r(x.real), self.ln_i(x.imag))

class ComplexGELU(nn.Module):
    def forward(self, x):
        return torch.complex(F.gelu(x.real), F.gelu(x.imag))

class ComplexDropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p
    def forward(self, x):
        if self.training:
            mask = (torch.rand(x.shape, device=x.device) > self.p).float() / (1.0 - self.p)
            return x * mask
        return x

# -----------------------------------------------------------
# 2. COMPONENTES POLYDIM (V5)
# -----------------------------------------------------------

class MagneticLaplacian(nn.Module):
    """Laplaciano Magnético que MANTIENE LA FASE COMPLEJA"""
    def __init__(self, N: int, q: float = math.pi / 4.0, skip_k: int = 3):
        super().__init__()
        self.N = N
        self.q = q
        
        A = self._build_adjacency(N, skip_k)
        A_s = 0.5 * (A + A.T)
        D_s = np.diag(np.sum(A_s, axis=1))
        Theta = 2 * np.pi * q * (A - A.T)
        H_q = A_s * np.exp(1j * Theta)
        L_q = D_s - H_q
        
        self.register_buffer('L_q_real', torch.from_numpy(L_q.real).float())
        self.register_buffer('L_q_imag', torch.from_numpy(L_q.imag).float())
        
    def _build_adjacency(self, N: int, skip_k: int) -> np.ndarray:
        A = np.zeros((N, N), dtype=np.float64)
        for i in range(N - 1):
            A[i, i + 1] = 1.0  # i mira a i+1 (espera... causalidad en matrices)
            # NOTA CRÍTICA CAUSAL: Si node i es el step T, y A[i, i-1] = 1, node i mira a i-1 (pasado).
            # Para causalidad estricta (GPT), el nodo actual i SÓLO puede recibir del pasado (j < i).
            # En la convención x_out = L @ x_in, y_i = sum_j L_{ij} x_j.
            # L_ij = - H_ij. Si A_ij = 1, H_ij != 0.
            # Por lo tanto, A debe ser LOWER TRIANGULAR para no mirar al futuro.
            # Fix V5 Causal: A_{i, i-1} = 1. (Nodo i recibe del i-1).
        
        A = np.zeros((N, N), dtype=np.float64)
        for i in range(1, N):
            A[i, i - 1] = 1.0  # Causal estricto local
        for k in range(2, skip_k + 1):
            for i in range(k, N):
                A[i, i - k] = 1.0 # Causal estricto con saltos
        return A
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T_seq = x.shape[-2]
        L_q = torch.complex(
            self.L_q_real[:T_seq, :T_seq], 
            self.L_q_imag[:T_seq, :T_seq]
        )
        if not torch.is_complex(x):
            x = x.to(torch.cfloat)
        # V5: NO hay proyección a real. Retornamos el tensor complejo.
        return torch.einsum('ij,...jd->...id', L_q, x)


class FourierEmbedding(nn.Module):
    def __init__(self, D: int, max_len: int = 5000):
        super().__init__()
        self.D = D
        freqs = torch.exp(torch.arange(0, D, 2).float() * (-math.log(10000.0) / D))
        self.freqs = nn.Parameter(freqs)
        
    def forward(self, positions: torch.Tensor) -> torch.Tensor:
        B, N = positions.shape
        angles = positions.unsqueeze(-1) * self.freqs.unsqueeze(0).unsqueeze(0)
        emb = torch.zeros(B, N, self.D, device=positions.device)
        emb[:, :, 0::2] = torch.sin(angles)
        emb[:, :, 1::2] = torch.cos(angles)
        # Lo convertimos a complejo (fase 0 en Y) para inyectar al sistema V5
        return emb.to(torch.cfloat)


class IsometricRotation(nn.Module):
    def __init__(self, D: int, use_fft: bool = True):
        super().__init__()
        self.D = D
        self.use_fft = use_fft
        self.phase = nn.Parameter(torch.randn(D) * 0.01)
        
    def forward(self, x: torch.Tensor, inverse: bool = False) -> torch.Tensor:
        sign = -1 if inverse else 1
        if self.use_fft:
            if not torch.is_complex(x): x = x.to(torch.cfloat)
            x_f = torch.fft.fft(x, dim=-1)
            x_f = x_f * torch.exp(1j * sign * self.phase).unsqueeze(0)
            x = torch.fft.ifft(x_f, dim=-1)
        return x


class PolydimLayerV5(nn.Module):
    def __init__(self, D: int, N: int, n_nodes: int, q: float, skip_k: int, dropout: float):
        super().__init__()
        self.D = D
        self.n_nodes = n_nodes
        self.d_node = D // n_nodes
        
        self.norm1 = ComplexLayerNorm(D)
        self.norm2 = ComplexLayerNorm(D)
        
        self.rotors = nn.ModuleList([
            IsometricRotation(self.d_node, use_fft=True)
            for _ in range(n_nodes)
        ])
        
        self.laplacian = MagneticLaplacian(N, q=q, skip_k=skip_k)
        
        self.mlp = nn.Sequential(
            ComplexLinear(D, 4 * D),
            ComplexGELU(),
            ComplexDropout(dropout),
            ComplexLinear(4 * D, D),
            ComplexDropout(dropout)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, seq_len, D = x.shape
        residual = x
        x = self.norm1(x)
        
        x_nodes = x.reshape(B, seq_len, self.n_nodes, self.d_node)
        rotated = []
        for i, rotor in enumerate(self.rotors):
            rotated.append(rotor(x_nodes[:, :, i, :]))
            
        x = torch.stack(rotated, dim=2).reshape(B, seq_len, D)
        x = self.laplacian(x)
        
        x = residual + x
        residual = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = residual + x
        return x


class PolydimMotorV5(nn.Module):
    def __init__(
        self,
        vocab_size: int = 50000,
        D: int = 256,
        N: int = 128,
        n_layers: int = 4,
        n_nodes: int = 4,
        q: float = math.pi / 4.0,
        skip_k: int = 3,
        dropout: float = 0.1
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.D = D
        
        self.token_embed = nn.Embedding(vocab_size, D)
        self.pos_embed = FourierEmbedding(D, max_len=N)
        self.dropout = ComplexDropout(dropout)
        
        self.layers = nn.ModuleList([
            PolydimLayerV5(D, N, n_nodes, q, skip_k, dropout)
            for _ in range(n_layers)
        ])
        
        self.norm = ComplexLayerNorm(D)
        # La salida proyecta concatenando Real e Imag a la dimensión original D
        # Esto porque D * 2 a vocab_size extrae la info de ambas partes.
        self.to_vocab = nn.Linear(D * 2, vocab_size)
        
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, seq_len = input_ids.shape
        # Embeddings en Real, se pasan a Complejo en FourierEmbedding
        token_emb = self.token_embed(input_ids).to(torch.cfloat)
        
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(B, -1)
        pos_emb = self.pos_embed(pos_ids)
        
        x = self.dropout(token_emb + pos_emb)
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.norm(x)
        
        # Colapsar a 1D/2D final extrayendo Real e Imag
        x_out = torch.cat([x.real, x.imag], dim=-1)
        logits = self.to_vocab(x_out)
        
        return logits

# -----------------------------------------------------------
# 3. ENTRENAMIENTO CONFIG
# -----------------------------------------------------------

CONFIG = {
    "D": 256,
    "N": 128,
    "n_layers": 4,
    "n_nodes": 4,
    "q": math.pi / 4.0,
    "skip_k": 3,
    "batch_size": 32,
    "block_size": 128,
    "max_iters": 5000,
    "eval_interval": 500,
    "learning_rate": 1e-3,
    "device": 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"\n[V5] Entrenando en: {CONFIG['device']}")

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - CONFIG['block_size'], (CONFIG['batch_size'],))
    x = torch.stack([d[i:i+CONFIG['block_size']] for i in ix])
    y = torch.stack([d[i+1:i+CONFIG['block_size']+1] for i in ix])
    return x.to(CONFIG['device']), y.to(CONFIG['device'])

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(50)
        for k in range(50):
            X, Y = get_batch(split)
            logits = model(X)
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), Y.view(B*T))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# -----------------------------------------------------------
# 4. RUN V5
# -----------------------------------------------------------
print("\n🚀 INICIANDO POLYDIM V5 (100% Complejo & Causal Estricto)...")
model_v5 = PolydimMotorV5(
    vocab_size=vocab_size, D=CONFIG['D'], N=CONFIG['N'],
    n_layers=CONFIG['n_layers'], n_nodes=CONFIG['n_nodes'],
    q=CONFIG['q'], skip_k=CONFIG['skip_k']
).to(CONFIG['device'])

optimizer = torch.optim.AdamW(model_v5.parameters(), lr=CONFIG['learning_rate'])

t0 = time.time()
for iter in range(CONFIG['max_iters'] + 1):
    if iter % CONFIG['eval_interval'] == 0:
        losses = estimate_loss(model_v5)
        print(f"Step {iter:5d} | train_loss {losses['train']:.4f} | val_loss {losses['val']:.4f} | time {time.time()-t0:.1f}s")
        
    xb, yb = get_batch('train')
    logits = model_v5(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("✅ Entrenamiento V5 finalizado.")

print("\n🧠 GENERANDO TEXTO CAUSAL V5...")
context = torch.zeros((1, 1), dtype=torch.long, device=CONFIG['device'])
model_v5.eval()
generated = []
for _ in range(500):
    cond = context[:, -CONFIG['block_size']:]
    logits = model_v5(cond)
    probs = F.softmax(logits[:, -1, :], dim=-1)
    next_tok = torch.multinomial(probs, num_samples=1)
    context = torch.cat((context, next_tok), dim=1)
    generated.append(next_tok.item())

print("\n--- V5 GENERADO ---")
print(decode(generated))

